# 18b - Follow-up: calibrate and set the threshold for the ensemble, to deploy it

**This is a follow-up to notebook 18, added after seeing its results, and reported as such.**

**Why:** in notebook 18 the ensemble (v2 ResNet-50 seed 44 + ConvNeXt-Tiny 1e-4 +
EfficientNet-B0, the members fixed in notebook 13) won D2: its AURC was 0.086 against 0.195
for the single model, with the whole bootstrap range below zero. It was not deployed only
because the backend served one model. That was an engineering limit, not a result: the
backend now averages several models, so the D2 winner is deployed, as D2 itself says.

**Rules, unchanged from notebook 18:** D6 (lowest out-of-sample log-loss among methods
that keep the diagnosis; a tie keeps temperature; the backend applies temperature, so another
winner would be reported and temperature deployed) and D7 as clarified in notebook 13b (the
lowest threshold whose bootstrap lower bound of accuracy on answered photos is at least 90%,
else the cost-based fallback). Nothing else is chosen here.

**How the ensemble is computed (the backend does exactly this):** each model's softmax
probabilities are averaged, the log of the average is divided by the temperature, and a
softmax gives the calibrated probabilities. The crop rule and threshold then apply as before.

Uses the scores notebooks 15 and 17 saved; no GPU needed. About 10 minutes.

In [ ]:
from pathlib import Path
import sys

from google.colab import drive

drive.mount("/content/drive")

REPO_URL = "https://github.com/BilliartMuthoni/confidence-calibrated-crop-diagnosis.git"
REPO_DIR = Path("/content/drive/MyDrive/Sproutly-app")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

ML_DIR = REPO_DIR / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

import config

In [ ]:
!git -C {REPO_DIR} config core.fileMode false
!git -C {REPO_DIR} pull --ff-only
!git -C {REPO_DIR} status --short --branch

In [ ]:
%pip install -q -r {ML_DIR}/requirements.txt

## Settings and the saved scores

In [ ]:
import importlib
import json
import shutil

import numpy as np
import pandas as pd
import torch

from src import calibration as cal, fusion, metrics, text_encoders, training
for module in (cal, fusion, metrics, text_encoders, training):
    importlib.reload(module)

TARGET, COST_WRONG, FLOOR, BOOTSTRAP = 0.90, 5.0, 0.5, 1000
FOLDS, REPEATS = 5, 3
SEED = config.RANDOM_SEED

N18 = ML_DIR / "results" / "d22_decision" / "decision.json"
decision18 = json.loads(N18.read_text())
MEMBERS = decision18["question_3_ensemble"]["ensemble"]
assert decision18["question_3_ensemble"]["ensemble_better"], "notebook 18 did not find the ensemble better"
assert MEMBERS == ["d17_resnet50_seed44", "d22_convnext_tiny_lr1e-04", "d22_efficientnet_b0"], MEMBERS
FOLDER = {"d17": config.MODELS_DIR / "d17_comparison", "d22": config.MODELS_DIR / "d22_comparison"}
V2 = [f"d17_resnet50_seed{s}" for s in (42, 43, 44)]
V3 = [f"d22_resnet50_seed{s}" for s in (42, 43, 44)]
SINGLE = decision18["deployed"]["model_run"]                       # notebook 18's single model, for comparison
LATE_HEAD = config.MODELS_DIR / "fusion_late_followup" / "late_head.pt"
OUT_DIR = config.MODELS_DIR / "ensemble_deployment"
REPO_RESULTS = ML_DIR / "results" / "ensemble_deployment"
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPO_RESULTS.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EVAL_SETS = ["val", "test", "ood_test", "selection"]
splits = {name: pd.read_csv(config.SPLITS_DIR / f"{name}.csv") for name in EVAL_SETS}
CLASSES = list(config.PLANTVILLAGE_CLASSES)
INDEX = {name: i for i, name in enumerate(CLASSES)}
CLASS_CROPS = np.array([{"Corn_(maize)": "maize", "Tomato": "tomato", "Potato": "potato"}[c.split("___")[0]]
                        for c in CLASSES])
SHORT = {c: c.replace("Corn_(maize)___", "Maize ").replace("___", " ").replace("_", " ").strip() for c in CLASSES}
FIELD_SOURCES = ["fieldplant", "tanzania"]

PARTS = {}
for name in EVAL_SETS:
    labels = splits[name]["class"].map(INDEX)
    keep = labels.notna().to_numpy()
    lab = labels[keep].astype(int).to_numpy()
    PARTS[name] = {"rows": splits[name][keep].reset_index(drop=True), "keep": keep, "labels": lab, "crops": CLASS_CROPS[lab]}
FIELD = {name: PARTS[name]["rows"]["source"].isin(FIELD_SOURCES).to_numpy() for name in ["val", "test"]}

SCORES = {}
for run in ["v1", *V2, *V3, *MEMBERS]:
    folder = FOLDER["d22"] if run.startswith("d22_") else FOLDER["d17"]
    SCORES[run] = {name: np.load(folder / f"{run}__{name}.npy")[PARTS[name]["keep"]] for name in EVAL_SETS}
# The ensemble's scores: log of the members' averaged probabilities (they act as logits).
SCORES["ensemble"] = {name: np.log(np.mean([metrics.softmax(SCORES[m][name]) for m in MEMBERS], axis=0) + 1e-12)
                      for name in EVAL_SETS}
print("Ensemble members:", MEMBERS, "| running on", DEVICE)

In [ ]:
def app(probs, name, mask=None):
    crops, labels = PARTS[name]["crops"], PARTS[name]["labels"]
    if mask is not None:
        crops, labels = crops[mask], labels[mask]
    conf, predicted = cal.crop_confidence(probs, CLASS_CROPS, crops)
    return conf, predicted, predicted == labels


def correct(run, name, mask=None):
    lg = SCORES[run][name] if mask is None else SCORES[run][name][mask]
    return app(metrics.softmax(lg), name, mask)[2]

## D6 and D7 for the ensemble

In [ ]:
sel = PARTS["selection"]
table = cal.compare(SCORES["ensemble"]["selection"], sel["labels"], folds=FOLDS, repeats=REPEATS, seed=SEED)
METHOD, REASON = cal.choose_method(table)
d6_summary = table.groupby("method").agg(nll=("nll", "mean"), nll_sd=("nll", "std"), ece=("ece", "mean"),
                                         aurc=("aurc", "mean"), changed_predictions=("changed_predictions", "mean"))
display(d6_summary.round(4))
DEPLOY_METHOD = METHOD if METHOD in ("temperature", "none") else "temperature"
print(f"D6: {METHOD} ({REASON}); deployed: {DEPLOY_METHOD}")

oof = cal.out_of_fold(DEPLOY_METHOD, SCORES["ensemble"]["selection"], sel["labels"], FOLDS, SEED)
conf_oof, _, correct_oof = app(oof, "selection")
D7 = cal.choose_threshold_verified(conf_oof, correct_oof, TARGET, COST_WRONG, FLOOR, n=BOOTSTRAP, seed=SEED)
THRESHOLD = D7["threshold"]
D7["answered_at_threshold"] = cal.at_threshold(conf_oof, correct_oof, THRESHOLD)
FINAL = cal.fit(DEPLOY_METHOD, SCORES["ensemble"]["selection"], sel["labels"])
TEMPERATURE = FINAL.params.get("temperature", 1.0)
print(json.dumps(D7, indent=1, default=float))
print(f"\nEnsemble: calibration {DEPLOY_METHOD} (T = {TEMPERATURE:.4f}) | threshold {THRESHOLD:.3f}")

## Ensemble against the single model of notebook 18, on every set

The single model's numbers are recomputed here the same way (its own calibration from
notebook 18), so the two rows of each set are directly comparable.

In [ ]:
single_cal = decision18["deployed"]
single_final = cal.Calibrator("temperature", {"temperature": single_cal["temperature"]})
single_oof = cal.out_of_fold("temperature", SCORES[SINGLE]["selection"], sel["labels"], FOLDS, SEED)


def row(probs, name, threshold, mask=None):
    conf, _, ok = app(probs, name, mask)
    at = cal.at_threshold(conf, ok, threshold)
    return {"photos": len(ok), "accuracy (crop given)": ok.mean(), "ece": metrics.expected_calibration_error(conf, ok),
            "aurc": cal.aurc(conf, ok), "answered": at["coverage"], "accuracy of answered": at["selective_accuracy"],
            "wrong and answered": at["wrong_and_answered"]}


rows = []
for label, name, mask in [("selection (out of fold)", "selection", None), ("test (in-domain)", "test", None),
                          ("test: field photos", "test", FIELD["test"]), ("PlantDoc (out-of-domain)", "ood_test", None)]:
    for model, calibrator, threshold, oof_probs in [(SINGLE, single_final, single_cal["threshold"], single_oof),
                                                    ("ensemble", FINAL, THRESHOLD, oof)]:
        lg = SCORES[model][name] if mask is None else SCORES[model][name][mask]
        probs = oof_probs if name == "selection" else calibrator.probabilities(lg)
        rows.append({"set": label, "model": model, **row(probs, name, threshold, mask)})
compare = pd.DataFrame(rows).set_index(["set", "model"])
display(compare.round(4))

## Where the ensemble gains and loses (reported only)

In [ ]:
def per_group(name, by):
    rows = PARTS[name]["rows"].assign(cls=lambda d: d["class"].map(SHORT))
    out = []
    for key, group in rows.groupby(by):
        mask = np.zeros(len(rows), bool)
        mask[group.index.to_numpy()] = True
        entry = dict(zip(by, key if isinstance(key, tuple) else (key,)))
        entry["photos"] = int(mask.sum())
        entry["v1"] = correct("v1", name, mask).mean()
        entry["v2 seed 44"] = correct(SINGLE, name, mask).mean()
        entry["v3 (mean of 3)"] = np.mean([correct(r, name, mask).mean() for r in V3])
        entry["ensemble"] = correct("ensemble", name, mask).mean()
        entry["ensemble vs v1"] = entry["ensemble"] - entry["v1"]
        out.append(entry)
    return pd.DataFrame(out).set_index(by)


selection_by_source_class = per_group("selection", ["source", "cls"])
plantdoc_by_class = per_group("ood_test", ["cls"])
test_by_class = per_group("test", ["cls"])
for title, t in [("Selection set by source and class", selection_by_source_class),
                 ("Test set by class", test_by_class), ("PlantDoc by class (reported only)", plantdoc_by_class)]:
    print(title)
    display(t.round(3))

In [ ]:
import matplotlib.pyplot as plt

INK, MUTED = "#0b0b0b", "#52514e"


def confusion(name, title, file):
    _, predicted, _ = app(metrics.softmax(SCORES["ensemble"][name]), name)
    labels = PARTS[name]["labels"]
    present = sorted(set(labels) | set(predicted))
    names = [SHORT[CLASSES[i]] for i in present]
    matrix = pd.crosstab(pd.Series([SHORT[CLASSES[i]] for i in labels], name="true class"),
                         pd.Series([SHORT[CLASSES[i]] for i in predicted], name="predicted (crop given)"))
    matrix = matrix.reindex(index=[n for n in names if n in matrix.index], columns=names, fill_value=0)
    shares = matrix.div(matrix.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(9, 7))
    ax.imshow(shares.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    for i in range(shares.shape[0]):
        for j in range(shares.shape[1]):
            if matrix.iat[i, j]:
                ax.text(j, i, matrix.iat[i, j], ha="center", va="center", fontsize=7,
                        color="white" if shares.iat[i, j] > 0.6 else INK)
    ax.set_xticks(range(len(shares.columns)), shares.columns, rotation=45, ha="right", fontsize=7, color=MUTED)
    ax.set_yticks(range(len(shares.index)), shares.index, fontsize=7, color=MUTED)
    ax.set_xlabel("Predicted (crop given)", color=INK, fontsize=9)
    ax.set_ylabel("True class", color=INK, fontsize=9)
    ax.set_title(title, color=INK, fontsize=10, loc="left")
    plt.tight_layout()
    plt.savefig(OUT_DIR / file, dpi=150)
    plt.show()
    matrix.to_csv(OUT_DIR / file.replace(".png", ".csv"))


confusion("selection", "Ensemble: selection set (new field sources), photo counts", "confusion_selection.png")
confusion("ood_test", "Ensemble: PlantDoc (reported only), photo counts", "confusion_plantdoc.png")

## Late fusion with the BERT text classifier, on the ensemble

In [ ]:
descriptions = pd.read_csv(ML_DIR / "data" / "symptoms" / "descriptions.csv", encoding="utf-8-sig")
descriptions = descriptions[descriptions["keep"] == "y"].reset_index(drop=True)
vectors = text_encoders.encode("bert_base", descriptions["text"].tolist(),
                               descriptions.loc[descriptions["split"] == "train", "text"].tolist(), DEVICE)
test_bank = fusion.TextBank.build(descriptions, vectors, CLASSES, "test")
head = fusion.TextClassifier(vectors.shape[1], int((~fusion.healthy_mask(CLASSES)).sum()))
head.load_state_dict(torch.load(LATE_HEAD, map_location="cpu"))
head.to(DEVICE).eval()


def features(model, name, mask=None):
    lg = SCORES[model][name] if mask is None else SCORES[model][name][mask]
    labels = PARTS[name]["labels"] if mask is None else PARTS[name]["labels"][mask]
    return fusion.Features(np.zeros((len(labels), 1), np.float32), lg, labels)


fusion_rows = []
for label, name, mask in [("test", "test", None), ("test: field", "test", FIELD["test"]), ("PlantDoc", "ood_test", None)]:
    part = features("ensemble", name, mask)
    fused = fusion.predict("late", head, part, *test_bank.fixed(part.labels), CLASSES, DEVICE)
    fusion_rows.append({"set": label, "condition": "ensemble alone", **metrics.summarise(part.logits, part.labels, len(CLASSES))})
    fusion_rows.append({"set": label, "condition": "late fusion, with test text", **metrics.summarise(fused, part.labels, len(CLASSES))})
fusion_table = pd.DataFrame(fusion_rows).set_index(["set", "condition"])
display(fusion_table.round(4))
FUSION_BIAS = fusion.bias_tests("late", head, features("ensemble", "test", FIELD["test"]), test_bank, CLASSES, DEVICE)
display(pd.Series(FUSION_BIAS).round(3))

## Save the decision and the backend settings

In [ ]:
calibrator = {"model_runs": MEMBERS, "model_version": "ensemble-v2r44-cnx-effb0",
              "combination": "mean of the members' softmax probabilities; log; divided by the temperature; softmax",
              "method": DEPLOY_METHOD, "temperature": TEMPERATURE, "threshold": THRESHOLD, "rule": D7["rule"],
              "fitted_on": f"selection set ({len(sel['labels'])} photos)",
              "confidence": "best class of the farmer's crop, 11-class probability, not rescaled"}
training.save_json(calibrator, OUT_DIR / "calibrator_final.json")
training.save_json({"why": "D2 winner in notebook 18; the backend's one-model limit was lifted", "D6": {"method": METHOD,
                    "reason": REASON}, "deployed_method": DEPLOY_METHOD, "D7": D7, "fusion_bias": FUSION_BIAS,
                    "deployed": calibrator, "post_hoc_followup": True}, OUT_DIR / "decision.json")
d6_summary.to_csv(OUT_DIR / "d6_methods.csv")
compare.to_csv(OUT_DIR / "ensemble_vs_single.csv")
selection_by_source_class.to_csv(OUT_DIR / "selection_by_source_class.csv")
test_by_class.to_csv(OUT_DIR / "test_by_class.csv")
plantdoc_by_class.to_csv(OUT_DIR / "plantdoc_by_class.csv")
fusion_table.to_csv(OUT_DIR / "late_fusion.csv")

print("Backend settings for the ensemble (put them in the backend's environment, not in Git):")
print("  ML_MODEL_PATH        = " + ",".join(f"models/{m}/model.pt" for m in MEMBERS))
print(f"  TEMPERATURE          = {TEMPERATURE:.4f}")
print(f"  CONFIDENCE_THRESHOLD = {THRESHOLD:.3f}")
print("  Copy each model.pt from Drive: " + ", ".join(f"CropDiseaseDiagnosis/models/{m}/" for m in MEMBERS))

## Copy the results into the repository, sync Drive, save to GitHub

Save the notebook first (Ctrl+S).

In [ ]:
for file in OUT_DIR.glob("*"):
    if file.suffix in {".csv", ".json", ".png"}:
        shutil.copy(file, REPO_RESULTS / file.name)
print(sorted(p.name for p in REPO_RESULTS.iterdir()))
drive.flush_and_unmount()
drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

from src import github_sync

shutil.copy(ML_DIR / "notebooks" / "18b_ensemble_deployment.ipynb", REPO_RESULTS / "notebook.ipynb")
print(github_sync.push(
    REPO_DIR,
    branch="colab/ensemble-deployment",
    message="Follow-up: calibration and threshold for the ensemble, to deploy it",
    token=userdata.get("GITHUB_TOKEN"),
    name="BilliartMuthoni",
    email=userdata.get("GIT_EMAIL"),
    paths=[str(REPO_RESULTS.relative_to(REPO_DIR))],
))